# 03 — Modélisation
Entraînement et comparaison de 3 modèles supervisés en cross-validation 5 folds.

**F1-score** : moyenne harmonique de la précision et du rappel. Il vaut 1 si le modèle ne fait aucune erreur, 0 s'il est nul. On préfère le F1 à la simple accuracy quand les classes sont déséquilibrées.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import joblib
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.model_selection import cross_validate, train_test_split
from sklearn.metrics import ConfusionMatrixDisplay, roc_curve, auc, f1_score, precision_score, recall_score

In [ ]:
df = pd.read_parquet('../data/twitter_clean.parquet')

FEATURES = [
    'statuses_count', 'followers_count', 'friends_count', 'favourites_count',
    'listed_count', 'default_profile', 'default_profile_image', 'geo_enabled',
    'profile_use_background_image', 'verified',
    'name_length', 'screen_name_length', 'description_length',
    'name_contains_bot', 'screen_name_contains_bot',
    'name_entropy', 'screen_name_entropy',
    'friends_followers_ratio', 'lists_followers_ratio',
    'retweet_followers_ratio', 'favorites_followers_ratio',
    'retweet_status_ratio', 'favorites_status_ratio', 'reply_status_ratio',
    'account_age',
    'followers_account_age_ratio', 'friends_account_age_ratio',
    'statuses_account_age_ratio', 'favourites_account_age_ratio',
    'lists_account_age_ratio',
]

X = df[FEATURES].fillna(0)
y = df['class_bot']
print(f'Dataset : {X.shape[0]} comptes, {X.shape[1]} features')
print(f'Bots : {y.sum()} ({y.mean()*100:.1f}%) | Humains : {(y==0).sum()}')

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
print('Train:', X_train.shape, '| Test:', X_test.shape)

## Définition des 3 modèles

In [ ]:
models = {
    'Régression Logistique': Pipeline([
        ('scaler', StandardScaler()),
        ('model', LogisticRegression(max_iter=1000, random_state=42)),
    ]),
    'Random Forest': Pipeline([
        ('scaler', StandardScaler()),
        ('model', RandomForestClassifier(n_estimators=100, random_state=42)),
    ]),
    'XGBoost': Pipeline([
        ('scaler', StandardScaler()),
        ('model', XGBClassifier(n_estimators=100, random_state=42, eval_metric='logloss', verbosity=0)),
    ]),
}
print('Modèles :', list(models.keys()))

## Cross-validation 5 folds
L'écart-type montre la stabilité du modèle : un faible écart-type signifie que le score est reproductible quel que soit le fold.

In [ ]:
scoring = ['f1', 'precision', 'recall', 'roc_auc']
results = {}
for name, model in models.items():
    cv = cross_validate(model, X_train, y_train, cv=5, scoring=scoring)
    results[name] = {
        m: {'mean': cv[f'test_{m}'].mean(), 'std': cv[f'test_{m}'].std()}
        for m in scoring
    }
    print(f'{name} — F1={results[name]["f1"]["mean"]:.3f} ± {results[name]["f1"]["std"]:.3f} | AUC={results[name]["roc_auc"]["mean"]:.3f} ± {results[name]["roc_auc"]["std"]:.3f}')

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
metrics = ['f1', 'precision', 'recall', 'roc_auc']
colors = ['steelblue', 'seagreen', 'crimson', 'orange']
x = np.arange(len(models))
width = 0.2

for i, (metric, color) in enumerate(zip(metrics, colors)):
    means = [results[m][metric]['mean'] for m in models]
    stds  = [results[m][metric]['std']  for m in models]
    ax.bar(x + i*width, means, width, label=metric, color=color, alpha=0.85,
           yerr=stds, capsize=4, error_kw={'elinewidth': 1.5, 'ecolor': 'black'})

ax.set_xticks(x + width * 1.5)
ax.set_xticklabels(list(models.keys()), rotation=10)
ax.set_ylim(0.5, 1.08)
ax.set_title('Comparaison CV 5 folds — moyenne ± écart-type')
ax.legend(loc='lower right')
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig('../plots/comparaison_modeles.png', dpi=120)
plt.show()

## Entraînement final

In [ ]:
model_keys = {'Régression Logistique': 'logistic_regression', 'Random Forest': 'random_forest', 'XGBoost': 'xgboost'}
for name, model in models.items():
    model.fit(X_train, y_train)
    joblib.dump(model, f'../models/{model_keys[name]}.joblib')
    print(f'Sauvegardé : {model_keys[name]}.joblib')

## Évaluation sur le jeu de test

In [ ]:
for name, model in models.items():
    y_pred = model.predict(X_test)
    print(f'{name} — F1={f1_score(y_test, y_pred):.3f} | Précision={precision_score(y_test, y_pred):.3f} | Rappel={recall_score(y_test, y_pred):.3f}')

## Feature importance — pourquoi 30 features ?
On part des 69 features disponibles. On entraîne XGBoost sur toutes, puis on garde celles dont l'importance est > 0. Ici, 30 features suffisent à capturer l'essentiel de l'information.

In [ ]:
importances = models['XGBoost'].named_steps['model'].feature_importances_
feat_df = pd.DataFrame({'feature': FEATURES, 'importance': importances}).sort_values('importance', ascending=False)
print(f'Features avec importance > 0 : {(feat_df["importance"] > 0).sum()} / {len(FEATURES)}')
feat_df.head(10)

## Top 10 features les plus importantes

In [ ]:
top10 = feat_df.head(10).sort_values('importance', ascending=True)
top10.plot.barh(x='feature', y='importance', figsize=(8, 5), color='steelblue', legend=False)
plt.title('Top 10 features — XGBoost')
plt.xlabel('Importance (gain)')
plt.tight_layout()
plt.savefig('../plots/feature_importance.png', dpi=120)
plt.show()
print('Feature #1 :', feat_df.iloc[0]['feature'])

## Matrices de confusion

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 4))
for ax, (name, model) in zip(axes, models.items()):
    ConfusionMatrixDisplay.from_estimator(model, X_test, y_test, ax=ax,
        display_labels=['Humain', 'Bot'], cmap=plt.cm.Blues)
    ax.set_title(name)
plt.tight_layout()
plt.savefig('../plots/confusion_matrices.png', dpi=120)
plt.show()

## Courbes ROC

In [ ]:
fig, ax = plt.subplots(figsize=(7, 6))
for name, model in models.items():
    y_score = model.predict_proba(X_test)[:, 1]
    fpr, tpr, _ = roc_curve(y_test, y_score)
    ax.plot(fpr, tpr, lw=2, label=f'{name} (AUC={auc(fpr, tpr):.3f})')
ax.plot([0,1],[0,1],'k--',lw=1)
ax.set_xlabel('Faux positifs')
ax.set_ylabel('Vrais positifs')
ax.set_title('Courbes ROC')
ax.legend()
plt.tight_layout()
plt.savefig('../plots/roc_curves.png', dpi=120)
plt.show()

## SHAP — interprétation locale
SHAP explique **pourquoi** le modèle a prédit bot ou humain pour un compte spécifique. Chaque barre montre la contribution d'une feature à la prédiction.

In [ ]:
import shap
explainer = shap.TreeExplainer(models['XGBoost'].named_steps['model'])
X_test_scaled = models['XGBoost'].named_steps['scaler'].transform(X_test)
shap_values = explainer.shap_values(X_test_scaled)
print('SHAP values calculées pour', len(X_test), 'comptes')

In [ ]:
# Interprétation locale — premier compte bot du jeu de test
idx_bot = y_test[y_test == 1].index[0]
pos = X_test.index.get_loc(idx_bot)
print(f'Compte analysé — label réel : {y_test.iloc[pos]} (1=bot)')
shap.waterfall_plot(
    shap.Explanation(values=shap_values[pos], base_values=explainer.expected_value,
                     data=X_test_scaled[pos], feature_names=FEATURES)
)